### **UNIVERSIDADE FEDERAL DE ITAJUBÁ**
### **Instituto de Matemática e Computação**
### **XMCO09 - Ciência de Redes** — Prof. Rafael Frinhani

## **Modelos de Referência (AT3) e Robustez (AT4) da rede de similaridade de estudantes**

---

**Pré-requisito:** executar antes o notebook `evasao_redes.ipynb` (gera `resultados/rede_baseline.pkl`).

| Seção | Conteúdo | CRISP-NET | Itens do formulário |
|---|---|---|---|
| 0 | Bibliotecas, parâmetros e funções auxiliares | — | — |
| 4 | Comparação com modelos clássicos (ER, WS, BA) | Fase 4 | AT3.01–3.12 |
| 5 | Robustez: falhas aleatórias × ataques direcionados | Fase 4 | AT4.02–4.12 |

Cada célula indica entre parênteses o item do formulário de avaliação que ela atende.

---

## **0. Bibliotecas, Parâmetros e Funções Auxiliares**

**Q0.1** Bibliotecas, parâmetros globais e carregamento da rede baseline.

In [1]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import igraph as ig
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DIR_FIG, DIR_RES = "figuras", "resultados"
os.makedirs(DIR_FIG, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES_MODELO = {"Empírica": "black", "ER": "#4c72b0", "WS": "#55a868", "BA": "#dd8452"}

gEmp = ig.Graph.Read_Pickle(os.path.join(DIR_RES, "rede_baseline.pkl"))
N, L = gEmp.vcount(), gEmp.ecount()
K_MEDIO = 2 * L / N
print(f"Rede baseline: N = {N} | L = {L} | <k> = {K_MEDIO:.2f}")

Rede baseline: N = 4424 | L = 178048 | <k> = 80.49


**Q0.2** Funções auxiliares (adaptadas dos gabaritos da AT3 e da AT4).

In [2]:
def salvar_fig(nome):
    caminho = os.path.join(DIR_FIG, nome + ".png")
    plt.savefig(caminho, dpi=300, bbox_inches="tight")
    print(f"Figura salva: {caminho}")


def distancias_amostradas(g, n_fontes=400, seed=SEED):
    """Caminho médio e diâmetro (estimado) no GCC via BFS a partir de n_fontes nós sorteados."""
    gi = g.connected_components().giant()
    if gi.vcount() < 2:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    fontes = rng.choice(gi.vcount(), size=min(n_fontes, gi.vcount()), replace=False)
    D = np.array(gi.distances(source=fontes.tolist()), dtype=float)
    finitas = D[np.isfinite(D) & (D > 0)]
    return finitas.mean(), finitas.max()


def metricas_rede(g):
    """Métricas estruturais usadas na comparação com os modelos (AT3)."""
    k = np.array(g.degree())
    dMedio, diam = distancias_amostradas(g)
    return {
        "k_medio": k.mean(),
        "densidade": g.density(),
        "k_max": k.max(),
        "heterogeneidade": (k ** 2).mean() / k.mean(),        # <k²>/<k>
        "clustering_medio": g.transitivity_avglocal_undirected(mode="zero"),
        "transitividade": g.transitivity_undirected(),
        "caminho_medio": dMedio,
        "diametro": diam,
        "gcc_frac": g.connected_components().giant().vcount() / g.vcount(),
        "assortatividade": g.assortativity_degree(directed=False),
    }


def distribuicao_graus(g):
    k = np.array(g.degree())
    valores, contagem = np.unique(k[k > 0], return_counts=True)
    return valores, contagem / len(k)

print("Funções auxiliares carregadas.")

Funções auxiliares carregadas.


---
## **4. Comparação com Modelos Clássicos de Redes** (CRISP-NET Fase 4 / AT3)

Redes sintéticas funcionam como **referências estatísticas**: todas têm o mesmo **N** e (aproximadamente) o mesmo **L** da rede baseline.

| Modelo | Hipótese estrutural testada | Parâmetros (calibração) |
|---|---|---|
| **ER** – Erdős–Rényi | Ligações ao acaso: Poisson, clustering ≈ densidade | *G(N, L)*: mesmo N e **mesmo L exato** |
| **WS** – Watts–Strogatz | Mundo pequeno: clustering alto e caminhos curtos | vizinhos *K* = ⟨k⟩; *p* calibrado numericamente para reproduzir o clustering médio empírico |
| **BA** – Barabási–Albert | Livre de escala: ligação preferencial gera *hubs* | *m* = ⟨k⟩/2 |

Distâncias (caminho médio e diâmetro) são estimadas por BFS a partir de 400 nós sorteados, igualmente para a rede empírica e para os modelos.

**Q4.1** *(AT3.01)* Caracterização estrutural da rede baseline (valores de referência).

In [3]:
metEmp = metricas_rede(gEmp)
refBaseline = pd.Series(metEmp, name="Empírica")
refBaseline.round(4).to_frame()

,Empírica
k_medio,80.4919
densidade,0.0182
k_max,349.0000
heterogeneidade,129.4159
clustering_medio,0.2985
transitividade,0.4019
caminho_medio,2.9548
diametro,5.0000
gcc_frac,1.0000
assortatividade,0.5180


**Q4.2** *(AT3.02–AT3.05)* Calibração dos parâmetros e geração de **30 réplicas** de cada modelo (sementes fixas).

In [4]:
N_REPLICAS = 30
NEI_WS = round(K_MEDIO / 2)                 # vizinhos de cada lado no anel (K = 2*nei)
M_BA = round(K_MEDIO / 2)                   # arestas por nó novo

P_WS = None                                 # definido pela calibração abaixo


def gerar(modelo, seed, p=None, m=None):
    random.seed(seed)                       # o igraph usa o gerador do módulo random
    if modelo == "ER":
        return ig.Graph.Erdos_Renyi(n=N, m=L)
    if modelo == "WS":
        g = ig.Graph.Watts_Strogatz(dim=1, size=N, nei=NEI_WS, p=P_WS if p is None else p)
        return g.simplify()
    if modelo == "BA":
        return ig.Graph.Barabasi(n=N, m=M_BA if m is None else m, directed=False)
    raise ValueError(modelo)


# Calibração do WS: p que reproduz o clustering médio empírico (busca em grade + interpolação linear)
gradeP = np.round(np.arange(0.02, 0.41, 0.02), 2)
cGrade = np.array([gerar("WS", SEED, p=p).transitivity_avglocal_undirected(mode="zero") for p in gradeP])
cAlvo = metEmp["clustering_medio"]
P_WS = float(np.interp(cAlvo, cGrade[::-1], gradeP[::-1]))   # C(p) é decrescente
print(f"WS: K = {2*NEI_WS} | clustering empírico = {cAlvo:.3f} -> p calibrado = {P_WS:.3f}")
print(f"BA: m = {M_BA}")

registros = []
t0 = time.time()
for modelo in ["ER", "WS", "BA"]:
    for r in range(N_REPLICAS):
        g = gerar(modelo, seed=SEED + r)
        registros.append({"modelo": modelo, "replica": r, "L": g.ecount(), **metricas_rede(g)})
    print(f"{modelo}: {N_REPLICAS} réplicas geradas ({time.time() - t0:.0f} s)")
replicas = pd.DataFrame(registros)
replicas.to_csv(os.path.join(DIR_RES, "at3_replicas.csv"), index=False)

print("\nComparabilidade (AT3.04): L médio por modelo vs empírico")
compL = replicas.groupby("modelo")["L"].mean().to_frame("L_medio")
compL["desvio_%"] = 100 * (compL["L_medio"] - L) / L
print(f"Empírica: L = {L}")
print(compL.round(2).to_string())

WS: K = 80 | clustering empírico = 0.298 -> p calibrado = 0.144
BA: m = 40


ER: 30 réplicas geradas (19 s)


WS: 30 réplicas geradas (36 s)


BA: 30 réplicas geradas (56 s)

Comparabilidade (AT3.04): L médio por modelo vs empírico
Empírica: L = 178048
         L_medio  desvio_%
modelo                    
BA      176140.0     -1.07
ER      178048.0      0.00
WS      176960.0     -0.61


**Q4.3** *(AT3.06, AT3.09)* Tabela comparativa (média ± desvio-padrão das 30 réplicas), erro relativo, *z-score* e escore de aderência.

In [5]:
METRICAS = ["k_medio", "densidade", "k_max", "heterogeneidade", "clustering_medio",
            "transitividade", "caminho_medio", "diametro", "gcc_frac", "assortatividade"]
METRICAS_ADER = ["heterogeneidade", "k_max", "clustering_medio", "transitividade",
                 "caminho_medio", "diametro", "assortatividade"]   # usadas no escore de aderência

media = replicas.groupby("modelo")[METRICAS].mean()
desvio = replicas.groupby("modelo")[METRICAS].std(ddof=1)

tabela = pd.DataFrame({"Empírica": refBaseline[METRICAS].round(3).astype(str)})
for mo in ["ER", "WS", "BA"]:
    tabela[mo] = [f"{media.loc[mo, c]:.3f} ± {desvio.loc[mo, c]:.3f}" for c in METRICAS]

erroRel = (100 * (media[METRICAS] - refBaseline[METRICAS]).abs() / refBaseline[METRICAS].abs()).T
zScore = ((refBaseline[METRICAS] - media[METRICAS]) / desvio[METRICAS].replace(0, np.nan)).T
aderencia = erroRel.loc[METRICAS_ADER].mean().sort_values()

tabela.to_csv(os.path.join(DIR_RES, "at3_tabela_comparativa.csv"))
erroRel.to_csv(os.path.join(DIR_RES, "at3_erro_relativo.csv"))
print("Tabela comparativa (média ± dp):")
print(tabela.to_string())
print("\nErro relativo (%) em relação à rede empírica:")
print(erroRel.round(1).to_string())
print("\nz-score (empírica - média do modelo) / dp:")
print(zScore.round(1).to_string())
print("\nEscore de aderência = média do erro relativo (%) nas métricas", METRICAS_ADER)
print(aderencia.round(1).to_string())
MODELO_ADERENTE = aderencia.index[0]
print(f"\n>>> Modelo mais aderente: {MODELO_ADERENTE} (erro médio = {aderencia.iloc[0]:.1f}%)")

Tabela comparativa (média ± dp):
                 Empírica               ER              WS                BA
k_medio            80.492   80.492 ± 0.000  80.000 ± 0.000    79.629 ± 0.000
densidade           0.018    0.018 ± 0.000   0.018 ± 0.000     0.018 ± 0.000
k_max               349.0  115.433 ± 2.861  97.400 ± 1.476  642.533 ± 25.887
heterogeneidade   129.416   81.480 ± 0.021  80.267 ± 0.006   142.311 ± 0.675
clustering_medio    0.298    0.018 ± 0.000   0.298 ± 0.001     0.057 ± 0.001
transitividade      0.402    0.018 ± 0.000   0.297 ± 0.001     0.059 ± 0.000
caminho_medio       2.955    2.209 ± 0.002   2.469 ± 0.002     2.200 ± 0.003
diametro              5.0    3.000 ± 0.000   3.000 ± 0.000     3.000 ± 0.000
gcc_frac              1.0    1.000 ± 0.000   1.000 ± 0.000     1.000 ± 0.000
assortatividade     0.518    0.000 ± 0.003  -0.000 ± 0.002     0.011 ± 0.002

Erro relativo (%) em relação à rede empírica:
modelo              BA    ER     WS
k_medio            1.1   0.0    0.6
d

**Q4.4** *(AT3.08)* Variabilidade das réplicas (boxplots) frente ao valor empírico (linha tracejada).

In [6]:
MET_BOX = ["clustering_medio", "transitividade", "caminho_medio",
           "diametro", "heterogeneidade", "assortatividade"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, met in zip(axes.ravel(), MET_BOX):
    dados = [replicas.loc[replicas["modelo"] == mo, met] for mo in ["ER", "WS", "BA"]]
    bp = ax.boxplot(dados, tick_labels=["ER", "WS", "BA"], patch_artist=True)
    for patch, mo in zip(bp["boxes"], ["ER", "WS", "BA"]):
        patch.set_facecolor(CORES_MODELO[mo]); patch.set_alpha(0.6)
    ax.axhline(refBaseline[met], color="black", ls="--", lw=1.5, label="Empírica")
    ax.set_title(met)
axes[0, 0].legend()
fig.suptitle(f"Rede empírica × {N_REPLICAS} réplicas de cada modelo", fontsize=13)
plt.tight_layout()
salvar_fig("4_4_boxplots_modelos")
plt.show()

Figura salva: figuras/4_4_boxplots_modelos.png


**Q4.5** *(AT3.07)* Distribuição de graus: rede empírica × modelos, em escala log-log (P(k) e distribuição acumulada complementar).

In [7]:
amostras = {"Empírica": gEmp, "ER": gerar("ER", SEED), "WS": gerar("WS", SEED), "BA": gerar("BA", SEED)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for nome, g in amostras.items():
    k, pk = distribuicao_graus(g)
    axes[0].loglog(k, pk, "o", ms=3, alpha=0.6, color=CORES_MODELO[nome], label=nome)
    graus = np.sort(np.array(g.degree()))
    ccdf = 1 - np.arange(len(graus)) / len(graus)
    axes[1].loglog(graus, ccdf, "-", lw=1.8, color=CORES_MODELO[nome], label=nome)
axes[0].set_xlabel("k"); axes[0].set_ylabel("P(k)"); axes[0].set_title("P(k) — log-log")
axes[1].set_xlabel("k"); axes[1].set_ylabel("P(K ≥ k)"); axes[1].set_title("CCDF — log-log")
for ax in axes:
    ax.legend()
plt.tight_layout()
salvar_fig("4_5_distribuicao_graus_modelos")
plt.show()

Figura salva: figuras/4_5_distribuicao_graus_modelos.png


**Q4.6** *(AT3.11)* Sensibilidade aos parâmetros dos modelos:
* **WS:** variação da probabilidade de religação *p* (curva clássica C(p)/C(0) e ⟨d⟩(p)/⟨d⟩(0)), com os valores empíricos como referência;
* **BA:** variação do número de ligações por nó *m* (altera ⟨k⟩).

In [8]:
N_REP_SENS = 3
valoresP = [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
valoresM = [20, 30, M_BA, 50, 60]

sensWS = []
for p in valoresP:
    for r in range(N_REP_SENS):
        g = gerar("WS", SEED + r, p=p)
        dm, _ = distancias_amostradas(g, n_fontes=200)
        sensWS.append({"p": p, "C": g.transitivity_avglocal_undirected(mode="zero"), "d": dm})
sensWS = pd.DataFrame(sensWS).groupby("p").mean()
g0 = gerar("WS", SEED, p=0.0)
C_0 = g0.transitivity_avglocal_undirected(mode="zero")
d_0, _ = distancias_amostradas(g0, n_fontes=200)

sensBA = []
for m in valoresM:
    for r in range(N_REP_SENS):
        g = gerar("BA", SEED + r, m=m)
        met = metricas_rede(g)
        sensBA.append({"m": m, "k_medio": met["k_medio"], "clustering_medio": met["clustering_medio"],
                       "caminho_medio": met["caminho_medio"], "heterogeneidade": met["heterogeneidade"],
                       "assortatividade": met["assortatividade"]})
sensBA = pd.DataFrame(sensBA).groupby("m").mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].semilogx(sensWS.index, sensWS["C"] / C_0, "o-", color=CORES_MODELO["WS"], label="C(p)/C(0)")
axes[0].semilogx(sensWS.index, sensWS["d"] / d_0, "s-", color="gray", label="<d>(p)/<d>(0)")
axes[0].axhline(metEmp["clustering_medio"] / C_0, color="black", ls="--", lw=1, label="C empírico / C(0)")
axes[0].axvline(P_WS, color=CORES_MODELO["WS"], ls=":", label=f"p calibrado = {P_WS:.2f}")
axes[0].set_xlabel("p (log)"); axes[0].set_title("WS: sensibilidade a p"); axes[0].legend(fontsize=8)
for met, mk in [("clustering_medio", "o-"), ("assortatividade", "s-")]:
    axes[1].plot(sensBA.index, sensBA[met], mk, label=f"BA – {met}")
    axes[1].axhline(metEmp[met], ls="--", lw=1, color=axes[1].lines[-1].get_color(), label=f"empírica – {met}")
axes[1].set_xlabel("m"); axes[1].set_title("BA: sensibilidade a m"); axes[1].legend(fontsize=8)
plt.tight_layout()
salvar_fig("4_6_sensibilidade_parametros")
plt.show()
print("WS:"); print(sensWS.round(3).to_string())
print("\nBA:"); print(sensBA.round(3).to_string())

Figura salva: figuras/4_6_sensibilidade_parametros.png
WS:
            C      d
p                   
0.0005  0.739  4.782
0.0010  0.736  4.156
0.0050  0.719  3.210
0.0100  0.698  2.971
0.0500  0.547  2.716
0.1000  0.397  2.559
0.2000  0.203  2.385
0.5000  0.028  2.228
1.0000  0.018  2.215

BA:
    k_medio  clustering_medio  caminho_medio  heterogeneidade  assortatividade
m                                                                             
20   39.905             0.034          2.591           77.400            0.006
30   59.790             0.046          2.362          110.627            0.008
40   79.629             0.057          2.201          142.088            0.011
50   99.424             0.067          2.097          172.305            0.014
60  119.173             0.077          2.033          201.556            0.016


**Q4.7** *(AT3.10, AT3.12, H2)* Síntese: coeficiente de mundo pequeno σ (Humphries & Gurney, 2008), modelo mais aderente e discrepâncias (métricas com |z| > 2 e erro > 30% em **todos** os modelos = propriedades que nenhum modelo clássico reproduz).

In [9]:
sigma = ((metEmp["clustering_medio"] / media.loc["ER", "clustering_medio"]) /
         (metEmp["caminho_medio"] / media.loc["ER", "caminho_medio"]))
print(f"Coeficiente de mundo pequeno σ = (C/C_ER)/(<d>/<d>_ER) = {sigma:.2f} "
      f"-> {'mundo pequeno (σ > 1)' if sigma > 1 else 'não é mundo pequeno'}  [H2]")
print(f"Modelo mais aderente: {MODELO_ADERENTE}")
print("\nMétricas reproduzidas (erro ≤ 30%) por modelo:")
for mo in ["ER", "WS", "BA"]:
    ok = erroRel.index[erroRel[mo] <= 30].tolist()
    print(f"  {mo}: {', '.join(ok) if ok else 'nenhuma'}")
nenhum = [c for c in METRICAS_ADER if (erroRel.loc[c] > 30).all() and (zScore.loc[c].abs() > 2).all()]
print("\nDiscrepâncias não capturadas por nenhum modelo:", ", ".join(nenhum) if nenhum else "nenhuma")
for c in nenhum:
    print(f"  {c}: empírica = {refBaseline[c]:.3f} | " +
          " | ".join(f"{mo} = {media.loc[mo, c]:.3f}" for mo in ["ER", "WS", "BA"]))

Coeficiente de mundo pequeno σ = (C/C_ER)/(<d>/<d>_ER) = 12.26 -> mundo pequeno (σ > 1)  [H2]
Modelo mais aderente: WS

Métricas reproduzidas (erro ≤ 30%) por modelo:
  ER: k_medio, densidade, caminho_medio, gcc_frac
  WS: k_medio, densidade, clustering_medio, transitividade, caminho_medio, gcc_frac
  BA: k_medio, densidade, heterogeneidade, caminho_medio, gcc_frac

Discrepâncias não capturadas por nenhum modelo: k_max, assortatividade
  k_max: empírica = 349.000 | ER = 115.433 | WS = 97.400 | BA = 642.533
  assortatividade: empírica = 0.518 | ER = 0.000 | WS = -0.000 | BA = 0.011


---
## **5. Robustez da Rede: Falhas Aleatórias × Ataques Direcionados** (CRISP-NET Fase 4 / AT4)

| Elemento | Definição neste trabalho |
|---|---|
| **Unidade de perturbação** | Nó (estudante): remover um nó = um perfil deixa de existir na rede |
| **Função preservada** | Conectividade entre perfis: fração **S** de nós no maior componente (GCC) |
| **Critério de falha** | **Fragmentação**: o GCC passa a conter **menos de 50% dos nós remanescentes**, $S_{rel} = S/(1-f) < 0{,}5$ (a rede deixa de ter um componente majoritário). O critério absoluto S < 10% de N seria trivial aqui: com f ≥ 0,9 restam no máximo 10% dos nós, mesmo sem nenhuma fragmentação |
| **Métricas monitoradas** | S (GCC / N), S_rel (GCC / nós restantes), nº de componentes, caminho médio no GCC (BFS amostrada) e fração do peso total remanescente (adaptação a rede ponderada) |

| Cenário | Regra de remoção |
|---|---|
| **A – Aleatória** | Nós sorteados uniformemente (30 réplicas) |
| **B – Grau** | Maior grau primeiro, **recalculado** a cada passo de 5% |
| **C – Betweenness** | Maior betweenness primeiro, recalculado a cada passo de 5% |
| **D – Evadidos** | Remoção dos nós com Target = Dropout (ordem aleatória, 30 réplicas), comparada ao cenário A |

A fração removida *f* varia de 0 a 0,95, em passos de 0,05. Falhas em cascata **não** são simuladas: a remoção de um estudante não altera a similaridade entre os demais (a rede é estática e derivada de atributos).

**Q5.1** *(AT4.03–AT4.08)* Funções de perturbação e medição (adaptadas de `remove_random_fraction` e `remove_top_fraction_by_degree` do gabarito da AT4).

In [10]:
FRACOES = np.round(np.arange(0.0, 0.96, 0.05), 2)
LIMIAR_FALHA = 0.50          # S_rel < 0,5 = fragmentação


def medir(h, N0, pesoTotal0, n_fontes=100):
    """Métricas de integridade após a remoção."""
    if h.vcount() == 0:
        return {"S": 0.0, "S_rel": 0.0, "componentes": 0, "d_medio": np.nan, "peso_rest": 0.0}
    comps = h.connected_components()
    gi = comps.giant()
    dm, _ = distancias_amostradas(gi, n_fontes=n_fontes) if gi.vcount() > 1 else (np.nan, np.nan)
    peso = sum(h.es["weight"]) / pesoTotal0 if "weight" in h.es.attributes() and pesoTotal0 else np.nan
    return {"S": gi.vcount() / N0, "S_rel": gi.vcount() / h.vcount(), "componentes": len(comps),
            "d_medio": dm, "peso_rest": peso}


def curva_ordem(g, ordem, fracoes=FRACOES):
    """Remove os nós na ordem dada (estratégia estática: aleatória ou por atributo)."""
    N0 = g.vcount()
    p0 = sum(g.es["weight"]) if "weight" in g.es.attributes() else None
    linhas = []
    for f in fracoes:
        k = int(round(f * N0))
        if k > len(ordem):
            break
        restantes = np.setdiff1d(np.arange(N0), ordem[:k], assume_unique=True)
        linhas.append({"f": f, **medir(g.induced_subgraph(restantes.tolist()), N0, p0)})
    return pd.DataFrame(linhas)


def curva_ataque(g, criterio, fracoes=FRACOES):
    """Ataque direcionado adaptativo: recalcula o critério a cada passo e remove os nós de maior valor."""
    N0 = g.vcount()
    p0 = sum(g.es["weight"]) if "weight" in g.es.attributes() else None
    h = g.copy()
    h.vs["id_orig"] = list(range(N0))
    removidos, ordemRem, linhas = 0, [], []
    for f in fracoes:
        alvo = int(round(f * N0))
        if alvo > removidos:
            score = h.degree() if criterio == "grau" else h.betweenness(directed=False)
            topo = np.argsort(score)[::-1][: alvo - removidos].tolist()
            ordemRem += [h.vs[v]["id_orig"] for v in topo]
            h.delete_vertices(topo)
            removidos = alvo
        linhas.append({"f": f, **medir(h, N0, p0)})
    return pd.DataFrame(linhas), ordemRem


def f_critico(curva, limiar=LIMIAR_FALHA, col="S_rel"):
    """Menor f com S_rel < limiar (interpolação linear); NaN se não houver colapso até f = 0,95."""
    curva = curva.reset_index(drop=True)
    abaixo = curva.index[curva[col] < limiar]
    if len(abaixo) == 0:
        return np.nan
    i = abaixo[0]
    if i == 0:
        return 0.0
    f1, s1, f2, s2 = curva.loc[i - 1, "f"], curva.loc[i - 1, col], curva.loc[i, "f"], curva.loc[i, col]
    return f1 + (s1 - limiar) * (f2 - f1) / (s1 - s2)

print("Funções de robustez carregadas.")

Funções de robustez carregadas.


**Q5.2** *(AT4.04–AT4.06)* Execução dos cenários A–D na rede baseline. ⏱️ *Cerca de 3 a 6 min.*

In [11]:
t0 = time.time()
rng = np.random.default_rng(SEED)

# A – Falhas aleatórias (30 réplicas)
curvasA = pd.concat([curva_ordem(gEmp, rng.permutation(N)).assign(replica=r) for r in range(N_REPLICAS)])
print(f"A (aleatória) concluído — {time.time() - t0:.0f} s")

# B – Ataque por grau (adaptativo)
curvaB, ordemB = curva_ataque(gEmp, "grau")
print(f"B (grau) concluído — {time.time() - t0:.0f} s")

# C – Ataque por betweenness (adaptativo)
curvaC, ordemC = curva_ataque(gEmp, "betweenness")
print(f"C (betweenness) concluído — {time.time() - t0:.0f} s")

# D – Remoção dos evadidos (ordem aleatória, 30 réplicas)
idxDrop = np.where(np.array(gEmp.vs["target"]) == "Dropout")[0]
fracDrop = len(idxDrop) / N
fracoesD = np.append(FRACOES[FRACOES < fracDrop], round(fracDrop, 4))
curvasD = pd.concat([curva_ordem(gEmp, rng.permutation(idxDrop), fracoes=fracoesD).assign(replica=r)
                     for r in range(N_REPLICAS)])
print(f"D (evadidos) concluído — {time.time() - t0:.0f} s | evadidos = {len(idxDrop)} ({100*fracDrop:.1f}% de N)")

resumoA = curvasA.groupby("f").agg(["mean", "std"])
resumoD = curvasD.groupby("f").agg(["mean", "std"])
pd.concat({"A_aleatoria": curvasA, "D_evadidos": curvasD}).to_csv(os.path.join(DIR_RES, "at4_curvas_estocasticas.csv"))
pd.concat({"B_grau": curvaB, "C_betweenness": curvaC}).to_csv(os.path.join(DIR_RES, "at4_curvas_ataques.csv"))

A (aleatória) concluído — 42 s


B (grau) concluído — 43 s


C (betweenness) concluído — 99 s


D (evadidos) concluído — 131 s | evadidos = 1421 (32.1% de N)


**Q5.3** *(AT4.08, AT4.09)* Curvas de degradação: S, caminho médio no GCC e peso remanescente em função de *f* (banda = IC 95% das 30 réplicas).

In [12]:
ic = lambda dp: 1.96 * dp / np.sqrt(N_REPLICAS)
fA = resumoA.index.values
fD = resumoD.index.values
estilos = {"A – Aleatória": "#4c72b0", "B – Grau": "#d62728", "C – Betweenness": "#9467bd", "D – Evadidos": "#ff7f0e"}

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, met, titulo in [(axes[0], "S", "S = GCC / N"),
                        (axes[1], "S_rel", "S_rel = GCC / nós restantes"),
                        (axes[2], "d_medio", "Caminho médio no GCC"),
                        (axes[3], "peso_rest", "Fração do peso total remanescente")]:
    m, s = resumoA[(met, "mean")], resumoA[(met, "std")]
    ax.plot(fA, m, "o-", ms=3, color=estilos["A – Aleatória"], label="A – Aleatória")
    ax.fill_between(fA, m - ic(s), m + ic(s), color=estilos["A – Aleatória"], alpha=0.25)
    ax.plot(curvaB["f"], curvaB[met], "s-", ms=3, color=estilos["B – Grau"], label="B – Grau")
    ax.plot(curvaC["f"], curvaC[met], "^-", ms=3, color=estilos["C – Betweenness"], label="C – Betweenness")
    mD, sD = resumoD[(met, "mean")], resumoD[(met, "std")]
    ax.plot(fD, mD, "d-", ms=4, color=estilos["D – Evadidos"], label="D – Evadidos")
    ax.fill_between(fD, mD - ic(sD), mD + ic(sD), color=estilos["D – Evadidos"], alpha=0.25)
    ax.set_xlabel("Fração de nós removidos f"); ax.set_title(titulo)
axes[1].axhline(LIMIAR_FALHA, color="black", ls="--", lw=1, label="Critério de falha (S_rel = 0,5)")
axes[1].legend(fontsize=8)
plt.tight_layout()
salvar_fig("5_3_curvas_degradacao")
plt.show()

fcA = f_critico(resumoA[("S_rel", "mean")].rename("S_rel").reset_index())
fcB, fcC = f_critico(curvaB), f_critico(curvaC)
fmtF = lambda x: f"{x:.3f}" if pd.notna(x) else "> 0,95 (sem colapso)"
print(f"f_c (S_rel < 0,5):  A = {fmtF(fcA)} | B = {fmtF(fcB)} | C = {fmtF(fcC)}")
sA_D = resumoA.loc[resumoA.index[np.argmin(np.abs(fA - 0.30))], ("S", "mean")]
sD_fim = resumoD[("S", "mean")].iloc[-1]
print(f"Cenário D: S após remover todos os evadidos (f = {fracDrop:.3f}) = {sD_fim:.3f} | "
      f"aleatória em f = 0,30: S = {sA_D:.3f}")

Figura salva: figuras/5_3_curvas_degradacao.png
f_c (S_rel < 0,5):  A = > 0,95 (sem colapso) | B = 0.816 | C = 0.597
Cenário D: S após remover todos os evadidos (f = 0.321) = 0.679 | aleatória em f = 0,30: S = 0.700


**Q5.4** *(AT4.11)* Critério de Molloy–Reed: limiar teórico de falhas aleatórias $f_c = 1 - \frac{1}{\kappa - 1}$, com $\kappa = \langle k^2 \rangle / \langle k \rangle$. A fórmula supõe rede sem correlações de grau; a diferença para o valor empírico indica o efeito da estrutura (clustering e assortatividade).

In [13]:
kappa = metEmp["heterogeneidade"]
fcTeorico = 1 - 1 / (kappa - 1)
kappaER = media.loc["ER", "heterogeneidade"]
print(f"κ empírico = {kappa:.2f} -> Molloy–Reed: rede possui componente gigante (κ > 2): {kappa > 2}")
print(f"f_c teórico (falhas aleatórias) = {fcTeorico:.3f}")
print(f"f_c empírico (cenário A, S_rel < 0,5) = {fmtF(fcA)}")
print(f"Referência ER: κ = {kappaER:.2f} -> f_c teórico = {1 - 1/(kappaER - 1):.3f}")

κ empírico = 129.42 -> Molloy–Reed: rede possui componente gigante (κ > 2): True
f_c teórico (falhas aleatórias) = 0.992
f_c empírico (cenário A, S_rel < 0,5) = > 0,95 (sem colapso)
Referência ER: κ = 81.48 -> f_c teórico = 0.988


**Q5.5** *(AT4.10)* Elementos críticos: nós mais centrais (com comunidade e *Target*) e impacto da remoção do 1% mais central.

In [14]:
dfNos = pd.read_csv(os.path.join(DIR_RES, "centralidades_nos.csv"))
critGrau = dfNos.nlargest(10, "grau")[["id", "grau", "betweenness", "comunidade", "target"]]
critBet = dfNos.nlargest(10, "betweenness")[["id", "betweenness", "grau", "comunidade", "target"]]
pd.concat({"grau": critGrau.reset_index(drop=True), "betweenness": critBet.reset_index(drop=True)}, axis=1) \
  .to_csv(os.path.join(DIR_RES, "at4_nos_criticos.csv"), index=False)
print("Top 10 por grau:");        print(critGrau.round(5).to_string(index=False))
print("\nTop 10 por betweenness:"); print(critBet.round(5).to_string(index=False))

k1 = int(round(0.01 * N))
top5 = dfNos.nlargest(int(round(0.05 * N)), "grau")
print(f"\nPerfil do top 5% por grau ({len(top5)} nós):")
print("  por comunidade:", top5["comunidade"].value_counts().to_dict())
print("  por Target:    ", top5["target"].value_counts().to_dict())

def s_apos(ordem):
    restantes = np.setdiff1d(np.arange(N), np.array(ordem[:k1]))
    return gEmp.induced_subgraph(restantes.tolist()).connected_components().giant().vcount() / N
print(f"\nS após remover 1% ({k1} nós): grau = {s_apos(ordemB):.3f} | betweenness = {s_apos(ordemC):.3f} | "
      f"aleatório = {curva_ordem(gEmp, rng.permutation(N), [0.01])['S'].iloc[0]:.3f}")

Top 10 por grau:
  id  grau  betweenness comunidade   target
 951   349      0.00250         C3  Dropout
2748   342      0.00184         C3  Dropout
3682   317      0.00094         C3  Dropout
 711   312      0.00118         C3  Dropout
 727   298      0.00076         C3  Dropout
3536   298      0.00090         C3  Dropout
2794   297      0.00120         C3 Enrolled
2274   294      0.00108         C3  Dropout
3103   294      0.00175         C3  Dropout
2640   293      0.00165         C3  Dropout

Top 10 por betweenness:
  id  betweenness  grau comunidade   target
4241      0.00653   125         C4  Dropout
1909      0.00307    96         C2 Graduate
3449      0.00296   162         C4 Graduate
3153      0.00294   119         C3 Graduate
2883      0.00274   133         C3 Graduate
3063      0.00273   107         C3 Graduate
1127      0.00267   102         C3 Graduate
3993      0.00264   110         C3 Graduate
 951      0.00250   349         C3  Dropout
4420      0.00248    94         C3


S após remover 1% (44 nós): grau = 0.990 | betweenness = 0.990 | aleatório = 0.990


**Q5.6** *(AT4.12)* Mesmo protocolo nos modelos da AT3 (falhas aleatórias: 5 réplicas; ataque por grau: 1 réplica) para relacionar a robustez observada à topologia.

In [15]:
N_REP_MOD = 5
robModelos = {}
for mo in ["ER", "WS", "BA"]:
    t0 = time.time()
    gA = [gerar(mo, SEED + r) for r in range(N_REP_MOD)]
    ca = pd.concat([curva_ordem(g, np.random.default_rng(SEED + r).permutation(N)).assign(replica=r)
                    for r, g in enumerate(gA)]).groupby("f")[["S", "S_rel"]].mean().reset_index()
    cb, _ = curva_ataque(gA[0], "grau")
    robModelos[mo] = {"A": ca, "B": cb}
    print(f"{mo}: concluído ({time.time() - t0:.0f} s)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(fA, resumoA[("S_rel", "mean")], "o-", ms=3, color="black", label="Empírica")
axes[1].plot(curvaB["f"], curvaB["S_rel"], "o-", ms=3, color="black", label="Empírica")
for mo in ["ER", "WS", "BA"]:
    axes[0].plot(robModelos[mo]["A"]["f"], robModelos[mo]["A"]["S_rel"], "-", color=CORES_MODELO[mo], label=mo)
    axes[1].plot(robModelos[mo]["B"]["f"], robModelos[mo]["B"]["S_rel"], "-", color=CORES_MODELO[mo], label=mo)
for ax, t in zip(axes, ["Falhas aleatórias (A)", "Ataque por grau (B)"]):
    ax.axhline(LIMIAR_FALHA, color="gray", ls="--", lw=1, label="critério de falha")
    ax.set_xlabel("f"); ax.set_ylabel("S_rel = GCC / nós restantes"); ax.set_title(t); ax.legend()
plt.tight_layout()
salvar_fig("5_6_robustez_modelos")
plt.show()

ER: concluído (8 s)


WS: concluído (7 s)


BA: concluído (7 s)


Figura salva: figuras/5_6_robustez_modelos.png


**Q5.7** *(AT4.09, AT4.12)* Resumo: limiar crítico $f_c$ (S_rel < 0,5) por rede e cenário.

In [16]:
resumoFc = pd.DataFrame({
    "A – Aleatória": [fcA] + [f_critico(robModelos[mo]["A"]) for mo in ["ER", "WS", "BA"]],
    "B – Grau": [fcB] + [f_critico(robModelos[mo]["B"]) for mo in ["ER", "WS", "BA"]],
    "C – Betweenness": [fcC, np.nan, np.nan, np.nan],
}, index=["Empírica", "ER", "WS", "BA"])
resumoFc.to_csv(os.path.join(DIR_RES, "at4_resumo_fc.csv"))
print("f_c (NaN = não colapsa até f = 0,95 ou não simulado):")
print(resumoFc.round(3).to_string())
print(f"\nMolloy–Reed (teórico, falhas aleatórias): {fcTeorico:.3f}")
print(f"Cenário D: S após remover os {len(idxDrop)} evadidos = {sD_fim:.3f}")

f_c (NaN = não colapsa até f = 0,95 ou não simulado):
          A – Aleatória  B – Grau  C – Betweenness
Empírica            NaN     0.816            0.597
ER                  NaN     0.922              NaN
WS                  NaN     0.925              NaN
BA                  NaN     0.820              NaN

Molloy–Reed (teórico, falhas aleatórias): 0.992
Cenário D: S após remover os 1421 evadidos = 0.679
